# Experiment: Stokes 粗细网格误差可视化 256

目标：
- 原始 PDE 数据由独立 Python 脚本生成，notebook 只负责读取、后处理和可视化。
- 将 coarse/fine 两套 FEM 混合系数投影到 `256 x 256` 公共网格。
- 在流体域 mask 内计算 coarse 相对 fine 的误差，并输出指标表与图像。

完成标准：
- 成功读取 `sample_params.npy`、`coarse/fine_mixed_coeff.npy` 和 `coarse/fine_mesh_info.npz`。
- 生成并缓存 `coarse_grid_256.npy`、`fine_grid_256.npy`、`valid_mask_256.npy`、`metrics.csv`、`metrics_summary.json` 与图像。


In [ ]:
# 环境、路径和 notebook 侧配置
from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
from scipy.sparse import csr_matrix
from scipy.spatial import Delaunay, cKDTree

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

ROOT = Path("/Users/zhougc/Desktop/IID/LRTOR_project/Bias_Aware_FNO")
ARTIFACT_DIR = ROOT / "output/jupyter-notebook/stokes-random-mesh-error-256"
FIG_DIR = ARTIFACT_DIR / "figures"
GEN_SCRIPT = ROOT / "scripts/generate_stokes_random_mesh_data.py"
NOTEBOOK_PATH = ROOT / "output/jupyter-notebook/stokes-random-mesh-error-256.ipynb"

for path in (ARTIFACT_DIR, FIG_DIR):
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "grid_size": 256,
    "gallery_seed": 123,
    "gallery_size": 6,
}

print(f"Notebook: {NOTEBOOK_PATH}")
print(f"Artifact dir: {ARTIFACT_DIR}")
print(f"Generation script: {GEN_SCRIPT}")
CONFIG


## 流程

1. 先运行原始数据生成脚本。
2. notebook 读取 coarse/fine 混合系数与网格信息。
3. 将两套系数投影到统一的 `256 x 256` 公共网格。
4. 在流体域 mask 内计算误差并保存图像。

推荐生成命令：
```bash
/Users/zhougc/miniconda3/envs/fenicsx-env/bin/python \
  /Users/zhougc/Desktop/IID/LRTOR_project/Bias_Aware_FNO/scripts/generate_stokes_random_mesh_data.py
```


In [ ]:
# notebook 侧辅助函数：读取缓存、插值和基础可视化
def to_serializable(value):
    if isinstance(value, dict):
        return {key: to_serializable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_serializable(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value


def save_json(path, payload):
    path.write_text(
        json.dumps(to_serializable(payload), indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )


def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def save_metrics_csv(path, rows):
    fieldnames = [
        "sample_idx",
        "alpha",
        "beta",
        "gamma",
        "rel_l2_all",
        "rel_l2_u_x",
        "rel_l2_u_y",
        "rel_l2_p",
        "rel_l2_velocity",
        "abs_err_mean",
        "abs_err_max",
    ]
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def load_metrics_csv(path):
    rows = []
    with path.open("r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            rows.append(
                {
                    "sample_idx": int(row["sample_idx"]),
                    "alpha": float(row["alpha"]),
                    "beta": float(row["beta"]),
                    "gamma": float(row["gamma"]),
                    "rel_l2_all": float(row["rel_l2_all"]),
                    "rel_l2_u_x": float(row["rel_l2_u_x"]),
                    "rel_l2_u_y": float(row["rel_l2_u_y"]),
                    "rel_l2_p": float(row["rel_l2_p"]),
                    "rel_l2_velocity": float(row["rel_l2_velocity"]),
                    "abs_err_mean": float(row["abs_err_mean"]),
                    "abs_err_max": float(row["abs_err_max"]),
                }
            )
    return rows


def load_npz_dict(path):
    data = np.load(path, allow_pickle=False)
    out = {key: data[key] for key in data.files}
    data.close()
    return out


def build_valid_mask(hole_centers, hole_radius, grid_size):
    gx = np.linspace(0.0, 1.0, int(grid_size), dtype=np.float64)
    gy = np.linspace(0.0, 1.0, int(grid_size), dtype=np.float64)
    xx, yy = np.meshgrid(gx, gy, indexing="ij")
    valid_mask = np.ones_like(xx, dtype=bool)
    r2 = float(hole_radius) ** 2
    for hx, hy in np.asarray(hole_centers, dtype=np.float64):
        valid_mask &= (xx - hx) ** 2 + (yy - hy) ** 2 >= r2
    return gx, gy, valid_mask


def build_sparse_interp_matrix(points, query_points, valid_mask_flat):
    tri = Delaunay(points)
    tree = cKDTree(points)

    n_query = int(query_points.shape[0])
    rows = []
    cols = []
    data = []

    valid_idx = np.flatnonzero(valid_mask_flat)
    valid_pts = query_points[valid_idx]
    simplex = tri.find_simplex(valid_pts)
    inside = simplex >= 0

    if np.any(inside):
        inside_rows = valid_idx[inside]
        inside_simplex = simplex[inside]
        transform = tri.transform[inside_simplex]
        delta = valid_pts[inside] - transform[:, 2]
        bary = np.einsum("nij,nj->ni", transform[:, :2], delta)
        weights = np.concatenate([bary, 1.0 - bary.sum(axis=1, keepdims=True)], axis=1)
        verts = tri.simplices[inside_simplex]
        rows.extend(np.repeat(inside_rows, 3).tolist())
        cols.extend(verts.reshape(-1).tolist())
        data.extend(weights.reshape(-1).tolist())

    outside = ~inside
    if np.any(outside):
        outside_rows = valid_idx[outside]
        nearest = tree.query(valid_pts[outside], k=1)[1]
        rows.extend(outside_rows.tolist())
        cols.extend(nearest.tolist())
        data.extend(np.ones(outside_rows.shape[0], dtype=np.float64).tolist())

    return csr_matrix((data, (rows, cols)), shape=(n_query, points.shape[0]), dtype=np.float64)


def masked_imshow(ax, values, valid_mask, title, cmap, vmin=None, vmax=None):
    masked = np.where(valid_mask, values, np.nan)
    image = ax.imshow(
        masked.T,
        origin="lower",
        extent=(0.0, 1.0, 0.0, 1.0),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    return image


In [ ]:
# notebook 侧后处理函数：公共网格投影、误差与图像输出
def project_mixed_coeff_to_grid(mixed_coeff, mesh_info, grid_size):
    mixed_coeff = np.asarray(mixed_coeff, dtype=np.float64)
    gx, gy, valid_mask = build_valid_mask(mesh_info["hole_centers"], mesh_info["hole_radius"], grid_size)
    xx, yy = np.meshgrid(gx, gy, indexing="ij")
    query_points = np.stack([xx.ravel(), yy.ravel()], axis=-1)
    valid_flat = valid_mask.reshape(-1)

    n_u = int(mesh_info["u_dof_count"])
    n_p = int(mesh_info["p_dof_count"])
    interp_u = build_sparse_interp_matrix(mesh_info["u_coords"], query_points, valid_flat)
    interp_p = build_sparse_interp_matrix(mesh_info["p_coords"], query_points, valid_flat)

    grid = np.empty((mixed_coeff.shape[0], int(grid_size), int(grid_size), 3), dtype=np.float32)
    for channel, coeff_slice, interp in (
        (0, slice(0, n_u), interp_u),
        (1, slice(n_u, 2 * n_u), interp_u),
        (2, slice(2 * n_u, 2 * n_u + n_p), interp_p),
    ):
        values = interp.dot(mixed_coeff[:, coeff_slice].T).T
        values[:, ~valid_flat] = 0.0
        grid[..., channel] = values.reshape(mixed_coeff.shape[0], int(grid_size), int(grid_size)).astype(np.float32)

    return grid, gx, gy, valid_mask


def compute_rel_l2(pred_flat, true_flat):
    diff = np.linalg.norm(pred_flat - true_flat, axis=1)
    denom = np.linalg.norm(true_flat, axis=1)
    rel = np.zeros_like(diff)
    np.divide(diff, denom, out=rel, where=denom > 1e-12)
    return rel


def compute_metrics_rows(params, coarse_grid, fine_grid, valid_mask):
    params = np.asarray(params, dtype=np.float64)
    coarse_grid = np.asarray(coarse_grid, dtype=np.float32)
    fine_grid = np.asarray(fine_grid, dtype=np.float32)
    valid_mask = np.asarray(valid_mask, dtype=bool)

    coarse_all = coarse_grid[:, valid_mask, :].reshape(coarse_grid.shape[0], -1)
    fine_all = fine_grid[:, valid_mask, :].reshape(fine_grid.shape[0], -1)
    rel_all = compute_rel_l2(coarse_all, fine_all)

    rel_channels = {}
    for channel_idx, channel_name in enumerate(["u_x", "u_y", "p"]):
        coarse_channel = coarse_grid[..., channel_idx][:, valid_mask].reshape(coarse_grid.shape[0], -1)
        fine_channel = fine_grid[..., channel_idx][:, valid_mask].reshape(fine_grid.shape[0], -1)
        rel_channels[channel_name] = compute_rel_l2(coarse_channel, fine_channel)

    coarse_velocity = coarse_grid[..., :2][:, valid_mask, :].reshape(coarse_grid.shape[0], -1)
    fine_velocity = fine_grid[..., :2][:, valid_mask, :].reshape(fine_grid.shape[0], -1)
    rel_velocity = compute_rel_l2(coarse_velocity, fine_velocity)

    abs_err = np.abs(coarse_grid - fine_grid)
    abs_err_flat = abs_err[:, valid_mask, :].reshape(abs_err.shape[0], -1)
    abs_err_mean = abs_err_flat.mean(axis=1)
    abs_err_max = abs_err_flat.max(axis=1)

    rows = []
    for idx, (alpha, beta, gamma) in enumerate(params):
        rows.append(
            {
                "sample_idx": int(idx),
                "alpha": float(alpha),
                "beta": float(beta),
                "gamma": float(gamma),
                "rel_l2_all": float(rel_all[idx]),
                "rel_l2_u_x": float(rel_channels["u_x"][idx]),
                "rel_l2_u_y": float(rel_channels["u_y"][idx]),
                "rel_l2_p": float(rel_channels["p"][idx]),
                "rel_l2_velocity": float(rel_velocity[idx]),
                "abs_err_mean": float(abs_err_mean[idx]),
                "abs_err_max": float(abs_err_max[idx]),
            }
        )

    top_idx = np.argsort(rel_all)[::-1][:5]
    summary = {
        "grid_size": int(coarse_grid.shape[1]),
        "n_samples": int(len(rows)),
        "rel_l2_all_mean": float(rel_all.mean()),
        "rel_l2_all_std": float(rel_all.std()),
        "rel_l2_velocity_mean": float(rel_velocity.mean()),
        "rel_l2_u_x_mean": float(rel_channels["u_x"].mean()),
        "rel_l2_u_y_mean": float(rel_channels["u_y"].mean()),
        "rel_l2_p_mean": float(rel_channels["p"].mean()),
        "abs_err_mean": float(abs_err_mean.mean()),
        "abs_err_max": float(abs_err_max.max()),
        "top5_samples": [int(i) for i in top_idx.tolist()],
    }
    return rows, summary


def save_mesh_comparison(coarse_mesh_info, fine_mesh_info, path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    for ax, mesh_info, title in (
        (
            axes[0],
            coarse_mesh_info,
            f"Coarse mesh ({int(coarse_mesh_info['cell_count'])} cells)",
        ),
        (
            axes[1],
            fine_mesh_info,
            f"Fine mesh ({int(fine_mesh_info['cell_count'])} cells)",
        ),
    ):
        triangulation = mtri.Triangulation(
            mesh_info["vertex_coords"][:, 0],
            mesh_info["vertex_coords"][:, 1],
            mesh_info["cell_vertices"],
        )
        ax.triplot(triangulation, color="0.15", linewidth=0.25)
        for hx, hy in np.asarray(mesh_info["hole_centers"], dtype=np.float64):
            circle = mpatches.Circle(
                (float(hx), float(hy)),
                float(mesh_info["hole_radius"]),
                facecolor="white",
                edgecolor="crimson",
                linewidth=1.0,
            )
            ax.add_patch(circle)
        ax.set_title(title)
        ax.set_aspect("equal")
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(0.0, 1.0)
        ax.set_xticks([])
        ax.set_yticks([])

    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def save_error_histogram(metrics_rows, path):
    values = np.asarray([row["rel_l2_all"] for row in metrics_rows], dtype=np.float64)
    fig, ax = plt.subplots(1, 1, figsize=(7, 4), constrained_layout=True)
    ax.hist(values, bins=18, color="#356D9A", edgecolor="white")
    ax.set_title("100 组参数的 masked rel_l2 分布")
    ax.set_xlabel("masked rel_l2(coarse vs fine)")
    ax.set_ylabel("样本数")
    ax.grid(True, alpha=0.25)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def save_field_triptych(reference, coarse, valid_mask, title_prefix, path, cmap="viridis"):
    reference = np.asarray(reference, dtype=np.float32)
    coarse = np.asarray(coarse, dtype=np.float32)
    err = np.abs(coarse - reference)
    ref_values = reference[valid_mask]
    coarse_values = coarse[valid_mask]
    vmin = float(min(ref_values.min(), coarse_values.min()))
    vmax = float(max(ref_values.max(), coarse_values.max()))

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), constrained_layout=True)
    im0 = masked_imshow(axes[0], reference, valid_mask, f"{title_prefix} | fine", cmap, vmin=vmin, vmax=vmax)
    im1 = masked_imshow(axes[1], coarse, valid_mask, f"{title_prefix} | coarse", cmap, vmin=vmin, vmax=vmax)
    im2 = masked_imshow(axes[2], err, valid_mask, f"{title_prefix} | |error|", "magma")
    fig.colorbar(im0, ax=axes[:2], fraction=0.03, pad=0.02)
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def save_random_galleries(coarse_grid, fine_grid, valid_mask, params, metrics_rows, fig_dir, gallery_size, gallery_seed):
    rng = np.random.default_rng(int(gallery_seed))
    sample_count = int(coarse_grid.shape[0])
    gallery_indices = np.sort(rng.choice(sample_count, size=min(int(gallery_size), sample_count), replace=False))

    for rank, sample_idx in enumerate(gallery_indices):
        sample_idx = int(sample_idx)
        alpha, beta, gamma = params[sample_idx]
        metric = metrics_rows[sample_idx]
        velocity_fine = np.linalg.norm(fine_grid[sample_idx, :, :, :2], axis=-1)
        velocity_coarse = np.linalg.norm(coarse_grid[sample_idx, :, :, :2], axis=-1)
        velocity_title = (
            f"sample {sample_idx:03d} | |u| | "
            f"a={alpha:.3f}, b={beta:.3f}, g={gamma:.3f} | rel_l2={metric['rel_l2_velocity']:.4f}"
        )
        save_field_triptych(
            velocity_fine,
            velocity_coarse,
            valid_mask,
            velocity_title,
            fig_dir / f"random_gallery_velocity_{rank:02d}_sample_{sample_idx:03d}.png",
            cmap="viridis",
        )

        pressure_title = (
            f"sample {sample_idx:03d} | p | "
            f"a={alpha:.3f}, b={beta:.3f}, g={gamma:.3f} | rel_l2={metric['rel_l2_p']:.4f}"
        )
        save_field_triptych(
            fine_grid[sample_idx, :, :, 2],
            coarse_grid[sample_idx, :, :, 2],
            valid_mask,
            pressure_title,
            fig_dir / f"random_gallery_pressure_{rank:02d}_sample_{sample_idx:03d}.png",
            cmap="coolwarm",
        )

    return [int(idx) for idx in gallery_indices.tolist()]


In [ ]:
# 读取原始数据，并在 notebook 内完成公共网格误差处理
sample_params_path = ARTIFACT_DIR / "sample_params.npy"
coarse_mixed_path = ARTIFACT_DIR / "coarse_mixed_coeff.npy"
fine_mixed_path = ARTIFACT_DIR / "fine_mixed_coeff.npy"
coarse_mesh_info_path = ARTIFACT_DIR / "coarse_mesh_info.npz"
fine_mesh_info_path = ARTIFACT_DIR / "fine_mesh_info.npz"
generation_summary_path = ARTIFACT_DIR / "generation_summary.json"
coarse_solver_stats_path = ARTIFACT_DIR / "coarse_solver_stats.npz"
fine_solver_stats_path = ARTIFACT_DIR / "fine_solver_stats.npz"

coarse_grid_path = ARTIFACT_DIR / f"coarse_grid_{CONFIG['grid_size']}.npy"
fine_grid_path = ARTIFACT_DIR / f"fine_grid_{CONFIG['grid_size']}.npy"
valid_mask_path = ARTIFACT_DIR / f"valid_mask_{CONFIG['grid_size']}.npy"
metrics_csv_path = ARTIFACT_DIR / "metrics.csv"
metrics_summary_path = ARTIFACT_DIR / "metrics_summary.json"

raw_required = [
    sample_params_path,
    coarse_mixed_path,
    fine_mixed_path,
    coarse_mesh_info_path,
    fine_mesh_info_path,
    generation_summary_path,
]
missing_raw = [path for path in raw_required if not path.exists()]
if missing_raw:
    cmd = (
        "/Users/zhougc/miniconda3/envs/fenicsx-env/bin/python "
        f"{GEN_SCRIPT}"
    )
    missing = "\n".join(str(path) for path in missing_raw)
    raise FileNotFoundError(
        "缺少原始数据，请先运行生成脚本。\n"
        f"命令: {cmd}\n"
        f"缺失文件:\n{missing}"
    )

sample_params = np.load(sample_params_path)
coarse_mixed = np.load(coarse_mixed_path)
fine_mixed = np.load(fine_mixed_path)
coarse_mesh_info = load_npz_dict(coarse_mesh_info_path)
fine_mesh_info = load_npz_dict(fine_mesh_info_path)
generation_summary = load_json(generation_summary_path)

if coarse_solver_stats_path.exists():
    coarse_solver_stats = load_npz_dict(coarse_solver_stats_path)
else:
    coarse_solver_stats = None
if fine_solver_stats_path.exists():
    fine_solver_stats = load_npz_dict(fine_solver_stats_path)
else:
    fine_solver_stats = None

if not np.allclose(coarse_mesh_info["hole_centers"], fine_mesh_info["hole_centers"]):
    raise RuntimeError("coarse/fine raw data 对应的孔洞几何不一致。")

derived_cached = all(path.exists() for path in [coarse_grid_path, fine_grid_path, valid_mask_path, metrics_csv_path, metrics_summary_path])
if derived_cached:
    print("使用已有的 notebook 后处理缓存。")
    coarse_grid = np.load(coarse_grid_path)
    fine_grid = np.load(fine_grid_path)
    valid_mask = np.load(valid_mask_path)
    metrics_rows = load_metrics_csv(metrics_csv_path)
    metrics_summary = load_json(metrics_summary_path)
else:
    print("开始执行 notebook 后处理：公共网格投影 + 误差计算。")
    coarse_grid, gx, gy, valid_mask = project_mixed_coeff_to_grid(coarse_mixed, coarse_mesh_info, CONFIG["grid_size"])
    fine_grid, gx_fine, gy_fine, valid_mask_fine = project_mixed_coeff_to_grid(fine_mixed, fine_mesh_info, CONFIG["grid_size"])
    if not np.array_equal(valid_mask, valid_mask_fine):
        raise RuntimeError("coarse/fine 生成的流体域 mask 不一致。")
    if not np.allclose(gx, gx_fine) or not np.allclose(gy, gy_fine):
        raise RuntimeError("coarse/fine 生成的公共网格坐标不一致。")

    metrics_rows, metrics_summary = compute_metrics_rows(sample_params, coarse_grid, fine_grid, valid_mask)
    metrics_summary.update(
        {
            "grid_size": int(CONFIG["grid_size"]),
            "raw_generation_summary": generation_summary,
            "valid_mask_true_count": int(valid_mask.sum()),
        }
    )
    np.save(coarse_grid_path, coarse_grid)
    np.save(fine_grid_path, fine_grid)
    np.save(valid_mask_path, valid_mask)
    save_metrics_csv(metrics_csv_path, metrics_rows)
    save_json(metrics_summary_path, metrics_summary)

assert sample_params.shape[1] == 3
assert coarse_grid.shape[0] == sample_params.shape[0]
assert fine_grid.shape == coarse_grid.shape
assert valid_mask.shape == (CONFIG["grid_size"], CONFIG["grid_size"])
assert len(metrics_rows) == sample_params.shape[0]

metrics_summary


In [ ]:
# 输出图像：网格对比、误差直方图和随机样本画廊
mesh_comparison_path = FIG_DIR / "mesh_comparison.png"
error_histogram_path = FIG_DIR / "error_histogram.png"

save_mesh_comparison(coarse_mesh_info, fine_mesh_info, mesh_comparison_path)
save_error_histogram(metrics_rows, error_histogram_path)
gallery_indices = save_random_galleries(
    coarse_grid,
    fine_grid,
    valid_mask,
    sample_params,
    metrics_rows,
    FIG_DIR,
    gallery_size=CONFIG["gallery_size"],
    gallery_seed=CONFIG["gallery_seed"],
)

figure_manifest = {
    "mesh_comparison": str(mesh_comparison_path),
    "error_histogram": str(error_histogram_path),
    "gallery_indices": gallery_indices,
}
save_json(ARTIFACT_DIR / "figure_manifest.json", figure_manifest)
figure_manifest


In [ ]:
# 文本摘要：误差最差样本前 10 名
top_rows = sorted(metrics_rows, key=lambda row: row["rel_l2_all"], reverse=True)[:10]
header = (
    f"{'sample':>6} {'alpha':>8} {'beta':>8} {'gamma':>8} "
    f"{'rel_l2':>10} {'vel_rel':>10} {'p_rel':>10} {'abs_mean':>10} {'abs_max':>10}"
)
lines = [header, "-" * len(header)]
for row in top_rows:
    lines.append(
        f"{row['sample_idx']:6d} {row['alpha']:8.3f} {row['beta']:8.3f} {row['gamma']:8.3f} "
        f"{row['rel_l2_all']:10.4f} {row['rel_l2_velocity']:10.4f} {row['rel_l2_p']:10.4f} "
        f"{row['abs_err_mean']:10.4e} {row['abs_err_max']:10.4e}"
    )

print("\n".join(lines))
print()
print(f"随机画廊样本: {gallery_indices}")
print(f"图片目录: {FIG_DIR}")
print(f"原始数据摘要: {generation_summary_path}")


## 说明

- PDE 原始数据由 [generate_stokes_random_mesh_data.py](/Users/zhougc/Desktop/IID/LRTOR_project/Bias_Aware_FNO/scripts/generate_stokes_random_mesh_data.py) 生成。
- notebook 只负责读取 raw data，执行公共网格投影、误差计算与可视化。
- 若重新生成 raw data，建议同时删除 `coarse_grid_256.npy`、`fine_grid_256.npy`、`valid_mask_256.npy`、`metrics.csv` 和 `metrics_summary.json`，避免使用旧的后处理缓存。
- 若只想做 smoke，可用脚本生成一个单独输出目录，再把 `ARTIFACT_DIR` 临时指向该目录。
